# Notebook 14 — Which genes we test for selection, and why

**Purpose.** This notebook is the *gene-panel justification* for the primate sexual-dichromatism selection
scan. For every gene whose selection we test across primates, it records **where we know it as a pigmentation
or sex-hormone gene from** — drawn only from **our own network layers** and the pigmentation references already
in this repository — and, for the proposed additions, whether the gene has **clean one-to-one orthologs across
primates** (a prerequisite for a trustworthy dN/dS scan).

**Why this notebook exists.** The original panel was built asymmetrically. The **hormone module** was assembled
as a *whole endocrine pathway* (53 genes: steroid biosynthesis, HPG axis, receptors, coactivators). The
**pigmentation module** was only the *canonical melanogenesis core* (27 genes). Because there are ~2x as many
hormone genes, raw per-module counts are hormone-heavy by construction, and any "module-balance" metric is
biased. The fix is not to trim hormones but to **expand pigmentation to comparable breadth** — using genes that
are pigmentation genes *in our own network layers*, not an imported external list.

**Evidence sources (all already in this repo).**
- **Raghunath melanogenesis mechanism** — the curated melanin-synthesis pathway layer (melanogenesis-pathway
  membership counts as pigmentation evidence).
- **OMIM pigmentation phenotype** — a gene whose loss/variation causes a documented human pigmentation disorder
  (albinism, Hermansky-Pudlak, Griscelli, vitiligo, hyper/hypopigmentation), annotated on our 803-node network.
- **Baxter et al. 2019** — a curated reference set of pigmentation genes (Baxter et al. 2019, *Pigment Cell
  Melanoma Res* 32:348, doi:10.1111/pcmr.12743), 657 human gene symbols.
- **Melanosome proteome (mass-spec)** and **functional CRISPR screen (Bajpai 2023)** — corroborating layers.

**What we do NOT add.** (i) Genes that entered the network only through generic STRING/KEGG proximity with no
pigmentation phenotype (pan-cellular kinases, apoptosis machinery). (ii) Genes without clean primate orthology
— see the orthology screen below.

In [ ]:
import os, pandas as pd
REPO = os.environ.get("PIGNET_REPO", os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))
CG   = os.path.join(REPO, "comparative-genomics")
DATA = os.path.join(CG, "analysis", "module_selection", "data")
J = pd.read_csv(os.path.join(DATA, "nb14_panel_justification.csv"))
print(f"genes documented: {len(J)}  |  already run: {(J.status=='already_run').sum()}  |  proposed: {(J.status=='to_run').sum()}")
print("by module:", J.module.value_counts().to_dict())

## 1 — The sex-hormone module (53 genes, already analyzed)

Justification = position in the endocrine pathway. The module spans the full hypothalamic-pituitary-gonadal
(HPG) axis and steroid metabolism, because sexual dichromatism is a sex-limited trait and the a-priori
hypothesis is that sex-steroid signaling gates its expression.

In [ ]:
hor = J[J.module == "hormone"]
print(hor.category.value_counts().to_string())
print("\ngenes:", ", ".join(sorted(hor.gene)))

## 2 — The pigmentation module, as originally run (27 genes)

The canonical melanogenesis core: synthesis enzymes (TYR, TYRP1, DCT), melanosome structural/transport
proteins, the MC1R/ASIP/POMC receptor-signaling module, and the MITF/SOX10/PAX3/TFAP2A transcription factors.
Every one is Baxter-listed and/or has an OMIM pigmentation phenotype.

In [ ]:
pig_run = J[(J.module == "pigmentation") & (J.status == "already_run")]
print(pig_run[["gene", "category", "justification"]].to_string(index=False))

## 3 — Candidate pigmentation expansion from our network layers (43 genes)

Two groups, both drawn from our own layers:

**(a) Melanosome biogenesis & transport (21)** — the single largest omission: the albinism / Hermansky-Pudlak /
Griscelli / vitiligo genes that build melanosomes and move them into keratinocytes (HPS1/3/4/5/6, AP-3 and
BLOC-1 complexes, the Griscelli tripod RAB27A/MYO5A/MLPH, LYST, GPR143). The original panel stopped at melanin
*synthesis* and omitted melanosome *biology*. Every gene has an OMIM pigmentation phenotype.

**(b) Melanogenesis regulation (22)** — the MAPK / Wnt / endothelin signaling arm that regulates MITF, from the
Raghunath melanogenesis-mechanism layer, each with corroborating pigmentation-specific evidence.

In [ ]:
add = J[J.status == "to_run"]
for grp in ["melanosome_biogenesis", "melanogenesis_regulation"]:
    sub = add[add.category == grp]
    print(f"\n{grp} ({len(sub)}):")
    print(sub[["gene", "justification"]].to_string(index=False))

## 4 — Orthology screen: can we recover clean orthologs across primates?

A dN/dS scan is only trustworthy if the gene has a **clean one-to-one ortholog** in each primate genome.
Two failure modes threaten that:

1. **Paralog mis-mapping** — the extraction step (miniprot) maps the human protein onto each genome and can
   silently land on a close **paralog**, corrupting the alignment and inflating dN/dS. Genes from large gene
   families are the danger.
2. **Incomplete ortholog coverage** — short or poorly annotated genes are not recovered in scaffold-level
   assemblies or the deeper (strepsirrhine) lineages, so they drop tips and lose power.

We screened all 43 candidates against **Ensembl Compara** (homology across 12 reference primates + human
within-species paralog count). The results are in the `n_primate_orth`, `n_within_paralog`, and `ortholog_risk`
columns of the justification table.

In [ ]:
scr = add.copy()
print("orthology risk of the 43 candidates:")
print(scr.ortholog_risk.value_counts().to_string())
print("\n--- FLAGGED genes (excluded) ---")
print(scr[scr.ortholog_risk != "clean"][["gene","category","n_primate_orth","n_within_paralog","ortholog_risk"]].to_string(index=False))

**The flagged genes fall into defensible-to-exclude categories:**
- `HIGH_paralog` — **FASLG** (7 human paralogs, TNF-ligand family), **SLC29A3** (SLC29 family). High
  mis-mapping risk.
- `low_ortholog_cov` — **BLOC1S3, BLOC1S5, HPS6** — small subunit genes recovered in <10/12 reference
  primates; expected to drop tips.
- `paralog_check` — BCL2, EDN1, MAP2K1, MAP2K2, MLPH, PAK1, RAC1, TP53 — 1–2 paralogs each. Note **MAP2K1 and
  MAP2K2 are mutually ~80% identical**, a specific cross-mapping hazard.

## 5 — The panel we run: the clean 30

Restricting to the **30 candidates with clean primate orthology** is not a compromise — it is better on *both*
axes. It fixes the asymmetry the expansion was meant to fix, and it does so with genes miniprot maps cleanly:

In [ ]:
clean = add[add.ortholog_risk == "clean"]
n_hor = 53
scenarios = [("current (broken)", 27), ("clean-only (run this)", 27+len(clean)), ("full 43 (not run)", 27+len(add))]
for label, npig in scenarios:
    bal = 2*npig/(npig+n_hor) - 1
    print(f"{label:24s}: pigmentation={npig:3d}  hormone={n_hor}  panel-neutral balance={bal:+.3f}")
print()
print(f"clean expansion: {len(clean)} genes "
      f"({(clean.category=='melanosome_biogenesis').sum()} melanosome + "
      f"{(clean.category=='melanogenesis_regulation').sum()} regulation)")
print("melanosome:", ", ".join(sorted(clean[clean.category=='melanosome_biogenesis'].gene)))
print("regulation:", ", ".join(sorted(clean[clean.category=='melanogenesis_regulation'].gene)))

The clean 30 lands the panel at **balance ≈ +0.04 — essentially symmetric** (57 pigmentation vs 53 hormone).
The full 43 would overshoot to +0.14 *and* carry the mis-mapping risk, so the flagged 13 are dropped: they cost
orthology reliability without improving balance. After the clean expansion runs, module balance should still be
reported as a **per-gene selection rate** (selected / genes-tested, per module) so it is robust to panel size.

## Result

A documented, orthology-screened panel. **80 genes already analyzed** (53 hormone + 27 pigmentation);
**30 pigmentation genes** with clean primate orthology queued for the selection scan, bringing pigmentation to
57 and the panel to near-symmetry. **13 candidates are excluded** for paralog/orthology risk and recorded as
such — an auditable exclusion, not a silent drop. Full table: `data/nb14_panel_justification.csv`; cluster list
with risk flags: `config/gene_panel_expansion_pigmentation.csv` (run the `ortholog_risk == "clean"` rows).